##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Function calling with Python

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Function_calling.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

 Function calling lets developers create a description of a function in their code, then pass that description to a language model in a request. The response from the model includes the name of a function that matches the description and the arguments to call it with. Function calling lets you use functions as tools in generative AI applications, and you can define more than one function within a single request.

This notebook provides code examples to help you get started. The documentation's [quickstart](https://ai.google.dev/gemini-api/docs/function-calling#python) is also a good place to start understanding function calling.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Function_calling.ipynb).

## Setup

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

Note: you may need to restart the kernel to use updated packages.


### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [11]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

### Choose a model

Function calling works on all Gemini models.

In [13]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-preview", "gemini-2.5-pro", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Setting up Functions as Tools

To use function calling, pass a list of functions to the `tools` parameter when creating an interaction. With the Interactions API, tools are declared as dictionaries. The SDK can also auto-generate tool schemas from Python functions.

In [16]:
def enable_lights():
    """Turn on the lighting system."""
    print("LIGHTBOT: Lights enabled.")


def set_light_color(rgb_hex: str):
    """Set the light color. Lights must be enabled for this to work."""
    print(f"LIGHTBOT: Lights set to {rgb_hex}.")

def stop_lights():
    """Stop flashing lights."""
    print("LIGHTBOT: Lights turned off.")

# Map function names to callables for execution
light_functions = {
    "enable_lights": enable_lights,
    "set_light_color": set_light_color,
    "stop_lights": stop_lights,
}

# For interactions.create, tools must be declared as dicts
light_controls = [
    {
        "type": "function",
        "name": "enable_lights",
        "description": "Turn on the lighting system.",
        "parameters": {"type": "object", "properties": {}},
    },
    {
        "type": "function",
        "name": "set_light_color",
        "description": "Set the light color. Lights must be enabled for this to work.",
        "parameters": {
            "type": "object",
            "properties": {
                "rgb_hex": {"type": "string", "description": "The color as an RGB hex string"}
            },
            "required": ["rgb_hex"],
        },
    },
    {
        "type": "function",
        "name": "stop_lights",
        "description": "Stop flashing lights.",
        "parameters": {"type": "object", "properties": {}},
    },
]

instruction = """
  You are a helpful lighting system bot. You can turn
  lights on and off, and you can set the color. Do not perform any
  other tasks.
"""

## Basic Function Calling with Multi-turn

Function calls naturally fit into multi-turn conversations. With the Interactions API, the model returns `function_call` outputs that you handle and respond to using `function_result` inputs.

**Note:** Automatic function calling (where the SDK calls your functions automatically) is a feature of `client.chats.create`. With `interactions.create`, you handle function calls manually, which gives you more control.
> The response from `interactions.create` contains a list of `steps`. For text responses, access the output via `interaction.steps[-1].content[0].text`.


In [19]:
# First turn: ask the model, providing tool declarations
interaction = client.interactions.create(
    model=MODEL_ID,
    input="It's awful dark in here...",
    system_instruction=instruction,
    tools=light_controls,
)

# Check if the model wants to call a function
for step in interaction.steps:
    if step.type == "function_call":
        print(f"Model wants to call: {step.name}({step.arguments})")
        fn = light_functions[step.name]
        result = fn(**step.arguments) if step.arguments else fn() if step.arguments else fn()

Model wants to call: enable_lights({})
LIGHTBOT: Lights enabled.


## Examining Function Calls and Execution History

With the Interactions API, the conversation state is managed server-side via `previous_interaction_id`. Each interaction object contains the outputs from that turn, which may include function calls, function results, or text responses.

In [21]:
from IPython.display import Markdown, display

# Show the outputs of the last interaction
for step in interaction.steps:
    print(f"Type: {step.type}")
    if hasattr(step, 'content') and step.content and step.content[0].text:
        print(f"Text: {step.content[0].text}")

Type: thought
Type: function_call


This history shows the flow:

1. **User**: Sends the message.

2. **Model**: Responds not with text, but with a `FunctionCall` requesting `enable_lights`.

3. **User (SDK)**: The `ChatSession` automatically executes `enable_lights()` because `automatic_function_calling` is enabled. It sends the result back as a `FunctionResponse`.

4. **Model**: Uses the function's result ("Lights enabled.") to formulate the final text response.

## Automatic Function Execution (`client.chats` only)

The `ChatSession` in the Python SDK (i.e., `client.chats.create`) has a powerful feature called Automatic Function Execution. When enabled (which it is by default), if the model responds with a FunctionCall, the SDK will:

1. Find the corresponding Python function in the provided `tools`.

2. Execute the function with the arguments provided by the model.

3. Send the function's return value back to the model in a `FunctionResponse`.

4. Return only the model's final response (usually text) to your code.

This significantly simplifies the workflow for common use cases.

> **Note:** Automatic function execution is only available with `client.chats.create` (which uses `generateContent` under the hood). With the Interactions API (`client.interactions.create`), you handle function calls manually — see [Manual function calling](#manual_fc) below.

**Example: Math Operations**

In [24]:
from google.genai import types

def add(a: float, b: float):
    """returns a + b."""
    print(f"  >> add({a}, {b})")
    return a + b

def subtract(a: float, b: float):
    """returns a - b."""
    print(f"  >> subtract({a}, {b})")
    return a - b

def multiply(a: float, b: float):
    """returns a * b."""
    print(f"  >> multiply({a}, {b})")
    return a * b

def divide(a: float, b: float):
    """returns a / b."""
    print(f"  >> divide({a}, {b})")
    if b == 0:
        return "Cannot divide by zero."
    return a / b

operation_tools = [add, subtract, multiply, divide]

chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": operation_tools,
        "automatic_function_calling": {"disable": False},  # Enabled by default
    },
)

response = chat.send_message(
    "I have 57 cats, each owns 44 mittens, how many mittens is that in total?"
)

print(response.text)

That is a total of 2,508 mittens.


In [25]:
# Print interaction outputs summary
for step in interaction.steps:
    print(f"Type: {step.type}", end="")
    if step.type == "model_output":
        print(f" | {step.content[0].text[:100]}")
    elif step.type == "function_call":
        print(f" | {step.name}({step.arguments})")
    else:
        print()

Type: thought
Type: function_call | enable_lights({})


Automatic execution handled the `multiply` call seamlessly.

## Automatic Function Schema Declaration

A key convenience of the Python SDK is its ability to automatically generate the required `FunctionDeclaration` schema from your Python functions. It inspects:

- **Function Name**: (`func.__name__`)

- **Docstring**: Used for the function's description.

- **Parameters**: Names and type annotations (`int`, `str`, `float`, `bool`, `list`, `dict`). Docstrings for parameters (if using specific formats like Google style) can also enhance the description.

- **Return Type Annotation**: Although not strictly used by the model for deciding which function to call, it's good practice.

You generally don't need to create `FunctionDeclaration` objects manually when using Python functions directly as tools.

However, you can generate the schema explicitly using `genai.types.FunctionDeclaration.from_callable` if you need to inspect it, modify it, or use it in scenarios where you don't have the Python function object readily available.

Use `.to_json_dict()` to get the JSON representation — this is the format you pass as tools when using `client.interactions.create`:

In [28]:
import json

set_color_declaration = types.FunctionDeclaration.from_callable(
    callable = set_light_color,
    client = client
)

print(json.dumps(set_color_declaration.to_json_dict(), indent=4))

{
    "description": "Set the light color. Lights must be enabled for this to work.",
    "name": "set_light_color",
    "parameters": {
        "properties": {
            "rgb_hex": {
                "type": "STRING"
            }
        },
        "required": [
            "rgb_hex"
        ],
        "type": "OBJECT"
    }
}


<a name="manual_fc"></a>
## Manual function calling

With the Interactions API (`client.interactions.create`), function calling is always manual — you receive function call requests from the model and handle them yourself. This gives you full control over execution, error handling, and validation.

Instead of chat sessions, the Interactions API uses `previous_interaction_id` to chain requests together, maintaining conversation context across turns.

The manual workflow works as follows:

1. Send a prompt with tool declarations to `client.interactions.create`.
2. Check the response `steps` for `function_call` entries.
3. Execute the corresponding functions locally.
4. Send the results back using `previous_interaction_id` to continue the conversation.

This would also be the case if you use a `Chat` with `"automatic_function_calling": {"disable": True}`.

**Example: Movies**

The following example is a rough equivalent of the [function calling single-turn curl sample](https://ai.google.dev/docs/function_calling#function-calling-single-turn-curl-sample) in Python. It uses functions that return (mock) movie playtime information, possibly from a hypothetical API:

In [32]:
def find_movies(description: str, location: str):
    """find movie titles currently playing in theaters."""
    return ["Barbie", "Oppenheimer"]


def find_theaters(location: str, movie: str):
    """Find theaters showing a movie in a location."""
    return ["Googleplex 16", "Android Theatre"]


def get_showtimes(location: str, movie: str, theater: str, date: str):
    """Find the start times for movies playing in a specific theater."""
    return ["10:00", "11:00"]

# Map function names to callables
available_functions = {
    "find_movies": find_movies,
    "find_theaters": find_theaters,
    "get_showtimes": get_showtimes,
}

# Dict-based tool declarations for interactions.create
theater_functions = [
    {"type": "function", "name": "find_movies",
     "description": "find movie titles currently playing in theaters based on any description, genre, title words, etc.",
     "parameters": {"type": "object", "properties": {
         "description": {"type": "string", "description": "Any kind of description including category or genre"},
         "location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}
     }, "required": ["description", "location"]}},
    {"type": "function", "name": "find_theaters",
     "description": "Find theaters based on location and movie title.",
     "parameters": {"type": "object", "properties": {
         "location": {"type": "string"}, "movie": {"type": "string"}
     }, "required": ["location", "movie"]}},
    {"type": "function", "name": "get_showtimes",
     "description": "Find the start times for movies playing in a specific theater.",
     "parameters": {"type": "object", "properties": {
         "location": {"type": "string"}, "movie": {"type": "string"},
         "theater": {"type": "string"}, "date": {"type": "string"}
     }, "required": ["location", "movie", "theater", "date"]}},
]

After using `interactions.create()` to ask a question, the model requests a `function_call`:

In [34]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Which theaters in Mountain View, CA show the Barbie movie?",
    tools=theater_functions,
)

# Find the function call in the outputs
for step in interaction.steps:
    if step.type == "function_call":
        print(f"Function call: {step.name}")
        print(f"Arguments: {step.arguments}")
        function_call_step = step
        break

Function call: find_theaters
Arguments: {'movie': 'Barbie', 'location': 'Mountain View, CA'}


Since this is not using a `ChatSession` with automatic function calling, you have to call the function yourself.

A very simple way to do this would be with `if` statements:

```python
if function_call.name == 'find_theaters':
  find_theaters(**function_call.args)
elif ...
```

However, since you already made the `theater_functions` list, this can be simplified to:


In [36]:
def call_function(function_call_output, functions):
    function_name = function_call_output.name
    function_args = function_call_output.arguments
    return functions[function_name](**function_args)

result = call_function(function_call_step, available_functions)
print(result)

['Googleplex 16', 'Android Theatre']


Finally, pass the function result back using `previous_interaction_id` to continue the conversation and get a final text response from the model.

In [38]:
# Send the function result back to the model
interaction_2 = client.interactions.create(
    model=MODEL_ID,
    previous_interaction_id=interaction.id,
    input={
        "type": "function_result",
        "name": function_call_step.name,
        "call_id": function_call_step.id,
        "result": str(result),
    },
    tools=theater_functions,
)

# Get the final text response
text_outputs = [o for o in interaction_2.steps if o.type == "text"]
if text_outputs:
    print(text_outputs[-1].text)

This demonstrates the manual workflow: call, check, execute, respond, call again.

## Parallel function calls

The Gemini API can call multiple functions in a single turn. This caters for scenarios where there are multiple function calls that can take place independently to complete a task.

First set the tools up. Unlike the movie example above, these functions do not require input from each other to be called so they should be good candidates for parallel calling.

In [41]:
def power_disco_ball(power: bool) -> bool:
    """Powers the spinning disco ball."""
    print(f"Disco ball is {'spinning!' if power else 'stopped.'}")
    return True

def start_music(energetic: bool, loud: bool, bpm: int) -> str:
    """Play some music matching the specified parameters."""
    print(f"Starting music! {energetic=} {loud=}, {bpm=}")
    return "Never gonna give you up."

def dim_lights(brightness: float) -> bool:
    """Dim the lights."""
    print(f"Lights are now set to {brightness:.0%}")
    return True

house_functions = {
    "power_disco_ball": power_disco_ball,
    "start_music": start_music,
    "dim_lights": dim_lights,
}

house_fns = [
    {"type": "function", "name": "power_disco_ball", "description": "Powers the spinning disco ball.",
     "parameters": {"type": "object", "properties": {"power": {"type": "boolean"}}, "required": ["power"]}},
    {"type": "function", "name": "start_music", "description": "Play some music matching the specified parameters.",
     "parameters": {"type": "object", "properties": {
         "energetic": {"type": "boolean"}, "loud": {"type": "boolean"}, "bpm": {"type": "integer"}
     }, "required": ["energetic", "loud", "bpm"]}},
    {"type": "function", "name": "dim_lights", "description": "Dim the lights.",
     "parameters": {"type": "object", "properties": {"brightness": {"type": "number"}}, "required": ["brightness"]}},
]

Now call the model with an instruction that could use all of the specified tools.

In [43]:
# Use "any" mode via tool_config is not yet supported in Interactions API.
# Instead, we make a direct call and handle functions manually.
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Turn this place into a party!",
    tools=house_fns,
)

prev_id = interaction.id
while any(s.type == "function_call" for s in interaction.steps):
    results = []
    for step in interaction.steps:
        if step.type == "function_call":
            print(f"Calling: {step.name}({step.arguments})")
            fn = house_functions[step.name]
            result = fn(**step.arguments) if step.arguments else fn() if step.arguments else fn()
            results.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": str(result),
            })
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=results,
        previous_interaction_id=prev_id,
        tools=house_fns,
    )
    prev_id = interaction.id

for step in interaction.steps:
    if step.type == "model_output":
        print(f"\nModel: {step.content[0].text}")

Calling: dim_lights({'brightness': 0.3})
Lights are now set to 30%
Calling: start_music({'bpm': 128, 'loud': True, 'energetic': True})
Starting music! energetic=True loud=True, bpm=128
Calling: power_disco_ball({'power': True})
Disco ball is spinning!

Model: OK! The lights are dimmed, the disco ball is spinning, and the music is cranking! Let's get this party started!


Notice the single model turn contains three FunctionCall parts, which the SDK then executed before getting the final text response.

## Compositional Function Calling
The model can chain function calls across multiple turns, using the result from one call to inform the next. This allows for complex, multi-step reasoning and task completion.

**Example: Finding Specific Movie Showtimes**

Let's reuse the theater_functions and ask a more complex query that requires finding movies first, then potentially theaters, then showtimes.

In [46]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="""
    Find comedy movies playing in Mountain View, CA on 01/01/2025.
    First, find the movie titles.
    Then, find the theaters showing those movies.
    Finally, find the showtimes for each movie at each theater.
""",
    tools=theater_functions,
)

prev_id = interaction.id
while any(s.type == "function_call" for s in interaction.steps):
    results = []
    for step in interaction.steps:
        if step.type == "function_call":
            print(f"Calling: {step.name}({step.arguments})")
            fn = available_functions[step.name]
            result = fn(**step.arguments) if step.arguments else fn()
            results.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": str(result),
            })
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=results,
        previous_interaction_id=prev_id,
        tools=theater_functions,
    )
    prev_id = interaction.id

for step in interaction.steps:
    if step.type == "model_output":
        print(step.content[0].text)

Calling: find_movies({'description': 'comedy', 'location': 'Mountain View, CA'})
Calling: find_theaters({'location': 'Mountain View, CA', 'movie': 'Barbie'})
Calling: find_theaters({'location': 'Mountain View, CA', 'movie': 'Oppenheimer'})
Calling: get_showtimes({'location': 'Mountain View, CA', 'date': '2025-01-01', 'theater': 'Googleplex 16', 'movie': 'Barbie'})
Calling: get_showtimes({'date': '2025-01-01', 'theater': 'Android Theatre', 'movie': 'Barbie', 'location': 'Mountain View, CA'})
Calling: get_showtimes({'date': '2025-01-01', 'movie': 'Oppenheimer', 'theater': 'Googleplex 16', 'location': 'Mountain View, CA'})
Calling: get_showtimes({'location': 'Mountain View, CA', 'date': '2025-01-01', 'theater': 'Android Theatre', 'movie': 'Oppenheimer'})
OK. On January 1, 2025, the comedy movies playing in Mountain View, CA are "Barbie" and "Oppenheimer". Both are showing at the Googleplex 16 and the Android Theatre. Here are the showtimes:

*   **Barbie**
    *   **Googleplex 16:** 10:00

Here you can see that the model made seven calls to answer your question and used the outputs of them in the subsequent calls and in the final answer.

## Function Calling Configuration using Modes

> **Note:** The `tool_config` modes (AUTO, ANY, NONE) are features of the `generateContent` API. They are **not yet available** with the Interactions API. The examples below use `client.interactions.create` to show equivalent workarounds.

With `generateContent`, you can precisely control when and which functions the model is allowed to call using the `tool_config` parameter.

The `tool_config` accepts a `ToolConfig` object, which contains a `FunctionCallingConfig` with two main fields:

- `mode`: Controls the overall function calling behavior (AUTO, ANY, NONE).

- `allowed_function_names`: An optional list of function names the model is restricted to calling in this turn.

Here's how each mode works and how to achieve similar behavior with the Interactions API:

### AUTO (Default Mode)

- **`generateContent` behavior**: The model decides whether to respond with text or to call one or more functions from the provided `tools`. This is the most flexible mode.

- **Interactions API equivalent**: This is the default behavior — when you pass `tools` to `interactions.create`, the model naturally decides whether to call them.

In [50]:
# In Interactions API, there is no "auto" mode config.
# The model will naturally decide whether to call tools.
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Turn on the lights!",
    system_instruction=instruction,
    tools=light_controls,
)

for step in interaction.steps:
    if step.type == "function_call":
        print(f"Function call: {step.name}({step.arguments})")
        fn = light_functions[step.name]
        result = fn(**step.arguments) if step.arguments else fn() if step.arguments else fn()
    elif step.type == "model_output":
        print(f"Model: {step.content[0].text}")

Function call: enable_lights({})
LIGHTBOT: Lights enabled.


### NONE Mode

- **`generateContent` behavior**: The model is explicitly prohibited from calling any functions, even if tools are provided. It will only respond with text.

- **Interactions API equivalent**: Simply don't pass any `tools` to the call.

In [52]:
# "none" mode equivalent: don't pass any tools
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Hello light-bot, what can you do?",
    system_instruction=instruction,
    # No tools provided = model can't call functions
)

for step in interaction.steps:
    if step.type == "model_output":
        print(f"Model: {step.content[0].text}")

Model: Hello! I am a lighting system bot. I can perform the following tasks for you:

*   **Turn lights on**
*   **Turn lights off**
*   **Set the light color**

How can I help you with your lighting today?


### ANY Mode

- **`generateContent` behavior**: Forces the model to call at least one function.

  - If `allowed_function_names` is set, the model must choose from that list.

  - If `allowed_function_names` is not set, the model must choose from the full tools list.

- **Interactions API equivalent**: There is no direct equivalent yet. Pass the tools and rely on the model's judgment. You can encourage function calling via system instructions or prompt phrasing.

Here's an example using the `generateContent` API with `tool_config`:

```python
from google.genai import types

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Make this place PURPLE!",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=light_controls,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="ANY",
            ),
        ),
    ),
)
```

With the Interactions API, you simply pass tools and handle calls manually:

In [54]:
# Force function calling by providing tools
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Make this place PURPLE!",
    system_instruction=instruction,
    tools=light_controls,
)

# Manual function execution loop
prev_id = interaction.id
while any(s.type == "function_call" for s in interaction.steps):
    results = []
    for step in interaction.steps:
        if step.type == "function_call":
            print(f"Calling: {step.name}({step.arguments})")
            fn = light_functions[step.name]
            result = fn(**step.arguments) if step.arguments else fn() if step.arguments else fn()
            results.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": str(result),
            })
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=results,
        previous_interaction_id=prev_id,
        tools=light_controls,
    )
    prev_id = interaction.id

for step in interaction.steps:
    if step.type == "model_output":
        print(f"Model: {step.content[0].text}")

Calling: enable_lights({})
LIGHTBOT: Lights enabled.
Calling: set_light_color({'rgb_hex': '#800080'})
LIGHTBOT: Lights set to #800080.
Model: OK. The lights are now purple!


## Next Steps
### Useful API references:

- The [genai.Client](https://googleapis.github.io/python-genai/genai.html#module-genai.client) class
  - Its [`client.interactions.create`](https://googleapis.github.io/python-genai/genai.html) method accepts `tools` as a list of function declarations or Python callables.
  - Its [`client.chats.create`](https://googleapis.github.io/python-genai/genai.html#module-genai.chats) method supports automatic function calling via `automatic_function_calling` config.
- With the **Interactions API**, the response contains `steps`. Steps of type `function_call` have `.name`, `.arguments`, and `.id` attributes. Return results using `previous_interaction_id` and `function_result` inputs.
- With **`generateContent`**, the response's [candidate](https://googleapis.github.io/python-genai/genai.html#genai.types.Candidate)'s [content](https://googleapis.github.io/python-genai/genai.html#genai.types.Content)'s [parts](https://googleapis.github.io/python-genai/genai.html#genai.types.Part) may contain a [genai.types.FunctionCall](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionCall).
- In response to a [FunctionCall](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionCall) the model always expects a [FunctionResponse](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionResponse).

### Related examples

Check out these examples using function calling to give you more ideas on how to use that very useful feature:
* [Barista Bot](../examples/Agents_Function_Calling_Barista_Bot.ipynb), an agent to order coffee
* [Browser-as-a-tool](../examples/Browser_as_a_tool.ipynb), using function calling to call a web-browser.
* Using function calling to [re-rank search results](../examples/Search_reranking_using_embeddings.ipynb).
* [Using tools with the Live API](../quickstarts/Get_started_LiveAPI_tools.ipynb), using function calling and other tools with the Live APIs.

### Continue your discovery of the Gemini API

Learn how to control how the Gemini API interact with your functions in the [function calling config](../quickstarts/Function_calling_config.ipynb) quickstart, discover how to control the model output in [JSON](../quickstarts/JSON_mode.ipynb) or using an [Enum](../quickstarts/Enum.ipynb) or learn how the Gemini API can generate and run code by itself using [Code execution](../quickstarts/Code_Execution.ipynb)